# LoRA & ControlNet

A refresher on two *adapter* techniques that adapt a frozen pretrained model by bolting on a small, trainable side-branch initialized to a **no-op** — **LoRA** for large language (and other transformer) models, **ControlNet** for diffusion image models.

**Domain:** LLM Inference, Training & Optimization  ·  **runnable:** yes

## 1. What & Why

Full fine-tuning of a large model means updating *every* weight and storing a full copy of the model per task. For a 7B-parameter LLM that is ~14 GB per task in fp16 — prohibitive in both GPU memory (you need optimizer states too) and storage. Both techniques here solve a version of *"adapt a big frozen model cheaply"*.

**LoRA (Low-Rank Adaptation)** freezes the pretrained weights `W` and learns a *low-rank* update `ΔW = B·A`, where `A` projects down to a small rank `r` and `B` projects back up. You train `A` and `B` (typically **<1%** of the parameters), get **MB-sized** checkpoints you can hot-swap per task, and — after merging `ΔW` back into `W` — pay **zero extra inference latency**. Reach for it whenever you want to specialize an LLM (or any big transformer) on a task or style without the cost of full fine-tuning. Don't bother if the base model is small enough to fully fine-tune, or if you need to inject large amounts of *new knowledge* (LoRA adapts behavior far better than it memorizes new facts).

**ControlNet** adds *spatial* conditioning to a text-to-image diffusion model: edge maps, depth, human pose, segmentation masks. Text prompts alone can't say "put the subject *here*, in *this* pose." ControlNet freezes the diffusion U-Net, **clones its encoder** into a trainable branch fed the control image, and wires that branch back in through **zero-initialized convolutions**. Reach for it when you need precise layout control over generation; skip it if a text prompt or simple img2img already gets you there.

**The common thread:** freeze a big pretrained trunk, attach a small trainable module, and initialize that module so it starts as a **no-op** (LoRA zeroes `B`; ControlNet zeroes its connecting convs). Training therefore *begins* from exactly the base model's behavior and only departs from it as far as the data demands — which makes training stable and avoids destroying the pretrained capability.

## 2. Mental Model

**LoRA — a blank sticky note on the recipe.** The frozen weight matrix `W` is a recipe you're not allowed to rewrite. Instead you stick a small note on top: `y = W·x + (alpha/r)·B·(A·x)`. `A` squeezes the input down to `r` dimensions (the note has limited room), `B` expands it back. Because `B` starts at **zero**, the note is blank at first and the output is *identical* to the base model — training gradually writes corrections onto it. A rank-`r` note on a `d×k` matrix costs `r·(d+k)` numbers instead of `d·k`; with `d=k=768, r=8` that's `12,288` vs `589,824` — a **48×** reduction.

**ControlNet — a photocopied encoder wired through dimmer switches.** Take the U-Net's encoder, photocopy it into a trainable branch, and feed the copy your control image (edges/depth/pose). Wire each copied block back into the original through a **zero-conv** — a dimmer switch turned fully off at the start. At step 0 the dimmers are off, so generation is exactly the base model. As training proceeds the dimmers open and inject spatial guidance, without ever having corrupted the frozen original.

```
LoRA:                                  ControlNet:
        x                                  noisy latent ──► [frozen U-Net encoder] ──► decoder
        │                                                          ▲
   ┌────┴─────┐                                                    │ zero-conv (off at init)
[frozen W]  [A]·r·[B←zero]                control image ──► [trainable encoder copy]
        │      │
        └──►(+)◄┘   ── same shape out
```

Both are: **frozen trunk + trainable side-branch that begins as a no-op.**

## 3. Key Concepts

- **Rank `r`** — the bottleneck dimension of the LoRA update. Higher `r` = more capacity *and* more parameters. Typical values 4–64; 8–16 covers most tasks.
- **`alpha` (scaling)** — the update is scaled by `alpha/r`, so `alpha` decouples the effective learning-rate of the adapter from its rank. A common convention is `alpha = 2·r` (or `alpha = r`). Change `r` and you often want to change `alpha` to keep the scale steady.
- **Target modules** — *which* layers get adapters. For transformers the attention projections (`q_proj`, `v_proj`, sometimes `k_proj`/`o_proj`) are the classic, cheapest choice; adapting all linear layers (incl. the MLP) costs more but can help.
- **Merging** — folding `ΔW = (alpha/r)·B·A` back into `W` gives a single matrix `W' = W + ΔW`. After merge, inference is indistinguishable in cost from the base model (no extra matmuls). Un-merged, you keep adapters swappable.
- **Adapter swapping / serving** — because adapters are tiny and additive, one base model can host many task adapters; serving stacks (e.g. S-LoRA, vLLM) swap them per request.
- **Zero-init (the no-op trick)** — LoRA zeroes `B`; ControlNet zeroes its connecting convs. Both make the side-branch contribute nothing at init, so the model starts as the pretrained one. This is the single most important shared design choice.
- **QLoRA** — LoRA on top of a 4-bit *quantized* frozen base, slashing memory further (see the QLoRA notebook). The base stays quantized; only the LoRA adapters are full-precision.
- **ControlNet conditioning** — the control image must be the *preprocessed* signal (a Canny edge map, a depth map, an OpenPose skeleton), produced by a detector — not a raw photo. The ControlNet weights are trained for a *specific* base model version (e.g. SD 1.5 vs SDXL) and a *specific* condition.

## 4. Setup

Core examples below need only **PyTorch** and run on CPU in seconds — we implement LoRA from scratch so the mechanics are explicit. The optional cells use Hugging Face **PEFT** (the production LoRA implementation) and **diffusers** (ControlNet); both are gated so the notebook runs without them.

```python
%pip install torch                 # core examples (CPU is fine)
%pip install peft transformers     # optional: production LoRA (Example 3)
%pip install diffusers controlnet-aux accelerate   # optional: ControlNet pipelines
```

In [1]:
import torch, torch.nn as nn

torch.manual_seed(0)
print("torch", torch.__version__, "| device: cpu (everything here is CPU-friendly)")

torch 2.12.1 | device: cpu (everything here is CPU-friendly)


## 5. Worked Examples

### Example 1 — LoRA from scratch: a frozen layer + a low-rank, zero-initialized adapter

We wrap a frozen `nn.Linear` with a LoRA update `(alpha/r)·B·A`. Because `B` is initialized to **zero**, the wrapped layer's output is *identical* to the frozen base at init — then we take one optimizer step and confirm only the tiny `A`/`B` matrices move.

In [2]:
class LoRALinear(nn.Module):
    """Frozen Linear + trainable low-rank update  y = W·x + (alpha/r)·B·A·x."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():      # freeze the pretrained weights
            p.requires_grad = False
        d_out, d_in = base.weight.shape
        self.r, self.scaling = r, alpha / r
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)  # down-projection
        self.B = nn.Parameter(torch.zeros(d_out, r))        # up-projection: ZERO -> no-op at init

    def forward(self, x):
        return self.base(x) + self.scaling * (x @ self.A.T) @ self.B.T


base = nn.Linear(768, 768, bias=False)
layer = LoRALinear(base, r=8, alpha=16)

x = torch.randn(4, 768)
with torch.no_grad():
    same = torch.allclose(layer(x), base(x))
print(f"Output identical to frozen base at init? {same}  (because B is zero)")

trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
print(f"Trainable params: {trainable:,}  |  Frozen params: {frozen:,}"
      f"  ->  {100*trainable/(trainable+frozen):.2f}% trainable")

Output identical to frozen base at init? True  (because B is zero)
Trainable params: 12,288  |  Frozen params: 589,824  ->  2.04% trainable


In [3]:
# One training step: only A and B carry gradients; the frozen base never moves.
opt = torch.optim.SGD([p for p in layer.parameters() if p.requires_grad], lr=0.1)
target = torch.randn(4, 768)

before_W = base.weight.detach().clone()
loss = ((layer(x) - target) ** 2).mean()
loss.backward()
print("grad on frozen base.weight :", base.weight.grad)        # None -> frozen
print("grad on adapter B present  :", layer.B.grad is not None)
opt.step()
print("frozen base.weight unchanged:", torch.equal(base.weight, before_W))
print("adapter now non-zero (B):    ", bool((layer.B.abs().sum() > 0).item()))

grad on frozen base.weight : None
grad on adapter B present  : True
frozen base.weight unchanged: True
adapter now non-zero (B):     True


### Example 2 — Merging is free, and the rank/parameter trade-off

Two facts that make LoRA cheap at inference and tunable at train time:

1. **Merging costs nothing at inference.** `y = W·x + s·B·A·x = (W + s·B·A)·x`, so we can fold the update into a single matrix `W' = W + s·B·A` and run one matmul — identical output, zero overhead. (This is also why you *can't* cleanly merge into a quantized base — `W` isn't a plain float matrix there.)
2. **Rank sets the parameter budget.** A LoRA update on a `d×d` matrix costs `2·d·r` params vs `d²` for full fine-tuning.

In [4]:
# 1) Merge the adapter into a single weight matrix and verify identical output.
with torch.no_grad():
    delta_W = layer.scaling * layer.B @ layer.A          # (768 x 768) low-rank update
    merged = nn.Linear(768, 768, bias=False)
    merged.weight.copy_(base.weight + delta_W)
    identical = torch.allclose(layer(x), merged(x), atol=1e-5)
print(f"Merged single-matmul output == two-path LoRA output? {identical}")
print(f"rank(ΔW) = {torch.linalg.matrix_rank(delta_W).item()} (<= r=8, by construction)\n")

# 2) Parameter savings vs full fine-tuning across ranks, for a 768x768 layer.
d = 768
full = d * d
print(f"{'rank':>4} | {'LoRA params':>12} | {'vs full FT':>10}")
for r in (1, 4, 8, 16, 64):
    lora = 2 * d * r
    print(f"{r:>4} | {lora:>12,} | {full/lora:>8.1f}x")

Merged single-matmul output == two-path LoRA output? True
rank(ΔW) = 4 (<= r=8, by construction)

rank |  LoRA params | vs full FT
   1 |        1,536 |    384.0x
   4 |        6,144 |     96.0x
   8 |       12,288 |     48.0x
  16 |       24,576 |     24.0x
  64 |       98,304 |      6.0x


### Example 3 (optional) — the same thing with Hugging Face PEFT

In production you don't hand-roll `LoRALinear`; you wrap a model with PEFT's `LoraConfig`. This cell is gated on PEFT being installed, so the notebook still runs without it — but it shows the real API and the trainable-parameter report you'd see on an actual model.

In [5]:
try:
    from peft import LoraConfig, get_peft_model

    # A tiny stand-in transformer block so this stays CPU-instant and download-free.
    toy = nn.Sequential()
    toy.add_module("q_proj", nn.Linear(256, 256))
    toy.add_module("v_proj", nn.Linear(256, 256))

    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                     lora_dropout=0.0, bias="none")
    peft_model = get_peft_model(toy, cfg)
    peft_model.print_trainable_parameters()
    print("PEFT injected adapters into:", [n for n, _ in peft_model.named_modules()
                                           if "lora_A" in n][:2])
except ImportError:
    print("peft not installed -- skipping. Install with: %pip install peft")
    print("API shape you'd run:")
    print("  cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'])")
    print("  model = get_peft_model(base_model, cfg)        # only adapters train")
    print("  ...train... ; model = model.merge_and_unload() # fold in for inference")

peft not installed -- skipping. Install with: %pip install peft
API shape you'd run:
  cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'])
  model = get_peft_model(base_model, cfg)        # only adapters train
  ...train... ; model = model.merge_and_unload() # fold in for inference


### Example 4 — ControlNet's zero-conv, in miniature

ControlNet's whole stability story is the **zero-convolution**: the trainable branch connects to the frozen U-Net through a conv whose weights *and* bias start at zero. At step 0 it outputs zeros, so the conditioned model is *exactly* the base model — then gradients flow and the connection "opens." Here's that mechanism in a few lines: a frozen base path plus a zero-conv'd control branch.

In [6]:
class TinyControlBlock(nn.Module):
    """Frozen base feature + control branch joined by a zero-initialized conv (the 'zero-conv')."""
    def __init__(self, ch=4):
        super().__init__()
        self.base = nn.Conv2d(ch, ch, 3, padding=1)        # frozen U-Net block (stand-in)
        for p in self.base.parameters():
            p.requires_grad = False
        self.control = nn.Conv2d(ch, ch, 3, padding=1)     # trainable encoder copy (stand-in)
        self.zero_conv = nn.Conv2d(ch, ch, 1)              # the connection
        nn.init.zeros_(self.zero_conv.weight)              # <-- zero weight
        nn.init.zeros_(self.zero_conv.bias)                # <-- zero bias

    def forward(self, latent, control_img):
        guidance = self.zero_conv(self.control(control_img))
        return self.base(latent) + guidance


block = TinyControlBlock()
latent      = torch.randn(1, 4, 8, 8)
control_img = torch.randn(1, 4, 8, 8)   # pretend: an edge/depth map encoded to latent space

with torch.no_grad():
    out_base = block.base(latent)
    out_full = block(latent, control_img)
print("At init, control branch contributes nothing? ",
      torch.allclose(out_base, out_full))
print("So generation starts == base model, regardless of the control image.")

# After a step the zero-conv 'opens' and guidance becomes non-zero.
opt = torch.optim.SGD([p for p in block.parameters() if p.requires_grad], lr=0.5)
loss = ((block(latent, control_img) - torch.randn(1, 4, 8, 8)) ** 2).mean()
loss.backward(); opt.step()
with torch.no_grad():
    moved = not torch.allclose(block.base(latent), block(latent, control_img))
print("After one step the connection has opened (guidance now non-zero)?", moved)

At init, control branch contributes nothing?  True
So generation starts == base model, regardless of the control image.
After one step the connection has opened (guidance now non-zero)? True


## 6. Gotchas & Pitfalls

- **`alpha` vs `r` confusion.** The effective update is scaled by `alpha/r`. If you raise `r` without touching `alpha`, the per-rank contribution shrinks; many recipes set `alpha = 2·r`. Forgetting this makes results inexplicably weak or unstable across ranks.
- **Rank too low / too high.** Too low (`r=1–2`) underfits expressive tasks; too high wastes parameters and can overfit small datasets. Start at `r=8–16` and only climb if validation says so.
- **Targeting the wrong modules.** Adapting only `q_proj`/`v_proj` is cheap and usually enough; if a task needs more capacity, add the MLP/`o_proj`. Putting LoRA on *embeddings* or the LM head is occasionally useful but often unnecessary cost.
- **Merging into a quantized base.** With QLoRA the frozen weights are 4-bit; you can't simply add a float `ΔW`. Either keep adapters un-merged at serve time, or dequantize-then-merge (losing some of the memory win). Plan for this before you ship.
- **Forgetting to freeze / eval the base.** If `requires_grad` isn't cleared on the base, you're silently full-fine-tuning (and blowing your memory budget). Double-check the trainable-parameter count.
- **Stacking multiple LoRAs.** Adapters are additive, so two LoRAs applied at once can interfere; weight them or merge deliberately rather than assuming they compose cleanly.
- **ControlNet: feeding a raw image as the condition.** The control input must be the *preprocessed* signal — a Canny map, depth map, or pose skeleton from a detector — not the original photo. Garbage condition ⇒ garbage control.
- **ControlNet: base-model / version mismatch.** A ControlNet trained for SD 1.5 won't work on SDXL, and a Canny ControlNet won't accept a depth map. Match the ControlNet to *both* the base model version and the condition type.
- **ControlNet guidance scale.** Pushing the conditioning weight too high freezes the layout but kills prompt adherence and image quality; too low and the control is ignored. It's a dial, not a switch.

## 7. When to Use vs Alternatives

**LoRA vs other ways to adapt an LLM**

| Approach | Trainable params | Memory | New knowledge | Notes |
|---|---|---|---|---|
| **Full fine-tuning** | 100% | Highest | Best | A full model copy per task; gold standard if you can afford it. |
| **LoRA** | <1% | Low | Limited | MB checkpoints, swappable, merge for zero-overhead inference. Default PEFT choice. |
| **QLoRA** | <1% | Lowest | Limited | LoRA on a 4-bit base; fits big models on one GPU. Can't merge cleanly. |
| **Prefix / prompt tuning** | Tiny | Lowest | Very limited | Learns soft prompt vectors; cheap but less expressive than LoRA. |
| **Adapters (Houlsby)** | ~1–3% | Low | Limited | Inserts bottleneck layers; adds inference latency (LoRA doesn't, once merged). |

Use **LoRA** as the default for task/style adaptation; **QLoRA** when memory is the binding constraint; **full FT** when you need maximum quality or to teach genuinely new capabilities and have the budget.

**ControlNet vs other ways to steer a diffusion model**

| Approach | Controls | Cost | When |
|---|---|---|---|
| **ControlNet** | Precise *spatial* layout (edge/depth/pose/seg) | Trained branch ≈ size of encoder | You need exact composition/structure. |
| **T2I-Adapter** | Similar spatial control | Much lighter than ControlNet | Good-enough control with far less compute. |
| **IP-Adapter** | *Style/identity* from a reference image | Light | "Make it look like this image," not "lay it out like this." |
| **img2img / inpainting** | Coarse layout from an init image | None (built-in) | Quick structural nudge; no precise control signal. |
| **DreamBooth / LoRA (on the diffusion model)** | A *subject* or *style* | Fine-tuning | Teach a new concept, not control one image's geometry. |

Reach for **ControlNet** when you need to pin down *where things go*; for style/identity transfer prefer **IP-Adapter**; for cheaper spatial control try **T2I-Adapter** first. Note these compose — a LoRA *and* a ControlNet can run on the same diffusion model at once.

## 8. Resources

- **LoRA paper** — *LoRA: Low-Rank Adaptation of Large Language Models* (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- **Hugging Face PEFT docs** — the production LoRA/adapter library: https://huggingface.co/docs/peft
- **QLoRA paper** — 4-bit base + LoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- **ControlNet paper** — *Adding Conditional Control to Text-to-Image Diffusion Models* (Zhang et al., 2023): https://arxiv.org/abs/2302.05543
- **diffusers ControlNet guide** — using ControlNet pipelines in practice: https://huggingface.co/docs/diffusers/using-diffusers/controlnet
- Related notebooks in this domain: **QLoRA** (4-bit base) and **TRL / RLHF & DPO** (what you often train *with* LoRA).